# Maximum projected extent versus delta tilt over an eddy lifetime

Choose **two random real eddies** and compare the two distances shown in `delta_tilt_schematic` throughout each eddy's available lifetime:

- **Maximum projected extent:** the range of the reference day's centres projected onto that day's fitted delta-tilt horizontal axis.
- **Delta `TiltDis`:** the horizontal endpoint separation of the reference-depth-limited delta fit, using a five-day Gaussian window with σ = 1 day.

Both measurements are recalculated from confirmed vertical profiles with the same helper/settings as the schematic, rather than reading a potentially older saved tilt table. This is **projected extent**, not maximum pairwise 2-D separation. The projection axis can change each day as the fitted tilt direction changes.

Change `SEED` for a different random pair, `N_EDDIES` for more examples, or set `EDDY_IDS` to choose specific eddies. Only candidate eddies are fitted until enough examples are found; no full-dataset fit scan is required.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

HERE = Path.cwd().resolve()
REPO = next(p for p in (HERE, *HERE.parents) if (p / 'seacofs_eddy_dataset_modular').is_dir())
FOLDER = REPO / 'seacofs_eddy_tilt_analysis' / 'delta_tilt_method'
sys.path.insert(0, str(FOLDER))
from delta_sensitivity_tools import scan_projected_extent_differences

PROFILE_PATH = Path('/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset_modular/vertical_profiles_confirmed/profiles.parquet')
SEED = 731
N_EDDIES = 2
EDDY_IDS = None             # e.g. [123, 456]; overrides N_EDDIES when supplied
MIN_VALID_DAYS = 10         # Minimum number of paired distance estimates per eddy
NUM = 5
TEMPORAL_SIGMA_DAYS = 1.0
DEPTH_INT = 10
MAX_DEPTH = 1000
MIN_DEPTH_RANGE = 200
MIN_POINTS = 5
BEARING_OFFSET = 20.0
SAVE = False
OUT = FOLDER / 'extent_lifetime_outputs'
if SAVE:
    OUT.mkdir(exist_ok=True)

In [ ]:
if not isinstance(N_EDDIES, int) or isinstance(N_EDDIES, bool) or N_EDDIES < 1:
    raise ValueError('N_EDDIES must be a positive integer.')
if not isinstance(MIN_VALID_DAYS, int) or MIN_VALID_DAYS < 1:
    raise ValueError('MIN_VALID_DAYS must be a positive integer.')
profiles = pd.read_parquet(PROFILE_PATH, columns=['Eddy', 'Day', 'Depth', 'xc', 'yc'])
if profiles[['Eddy', 'Day']].isna().any().any():
    raise ValueError('Missing Eddy/Day identifiers in the profile table.')
counts = profiles.groupby('Eddy').Day.nunique()
if EDDY_IDS is None:
    # Uniform random order of eddy IDs; selection does not depend on distance mismatch.
    candidates = np.random.default_rng(SEED).permutation(counts[counts >= MIN_VALID_DAYS].index)
    requested = N_EDDIES
else:
    candidates = list(dict.fromkeys(EDDY_IDS))
    requested = len(candidates)
    if not candidates:
        raise ValueError('EDDY_IDS must contain at least one ID, or be None.')
    missing = set(candidates) - set(counts.index)
    if missing:
        raise ValueError(f'Eddy IDs absent from profiles: {sorted(missing)}')

settings = dict(depth_int=DEPTH_INT, max_depth=MAX_DEPTH, sigma=TEMPORAL_SIGMA_DAYS,
    half_window=NUM // 2, min_depth_range=MIN_DEPTH_RANGE,
    min_points=MIN_POINTS, bearing_offset=BEARING_OFFSET)
series, selected = [], []
for eddy in candidates:
    eddy = int(eddy)
    track = profiles.loc[profiles.Eddy.eq(eddy)]
    measured = scan_projected_extent_differences(track, **settings).drop(columns='Rank')
    if len(measured) < MIN_VALID_DAYS:
        if EDDY_IDS is not None:
            print(f'Skipping eddy {eddy}: {len(measured)} valid pairs; requires {MIN_VALID_DAYS}.')
        continue
    # Reindex on all calendar days so missing estimates create visible gaps.
    days = pd.DataFrame({'Day': np.arange(int(track.Day.min()), int(track.Day.max()) + 1)})
    daily = days.merge(measured, on='Day', how='left', validate='one_to_one')
    daily['Eddy'] = eddy
    daily['AgeDays'] = daily.Day - int(track.Day.min())
    series.append(daily)
    selected.append(eddy)
    print(f'Selected eddy {eddy}: {len(measured)} paired days over {len(daily)} calendar days')
    if len(selected) == requested:
        break
if not series:
    raise ValueError('No eligible eddies. Reduce MIN_VALID_DAYS or choose different EDDY_IDS.')
if len(selected) < requested:
    print(f'Found {len(selected)} eligible eddies; requested {requested}.')
comparison = pd.concat(series, ignore_index=True)
print(f'To reproduce this selection: EDDY_IDS = {selected}')

## Lifetime comparison

The upper panel shows the two distances in kilometres. The lower panel shows **projected extent − delta `TiltDis`**: positive values mean projected extent is larger. Gaps remain gaps; the plotted curves receive no additional smoothing.

Age starts at the first available confirmed vertical profile, not necessarily physical formation or first surface detection. The first/last two calendar days lack a centred five-day fit. Missing reference days, insufficient depth support and zero horizontal tilt (undefined projection axis) have no paired estimate. Neighbouring days can contribute only within the reference day's supported depth intervals. The available reference depth may vary over the lifetime, so changes can reflect both structure and coverage.

In [ ]:
for eddy, daily in comparison.groupby('Eddy', sort=False):
    valid = daily.dropna(subset=['MaxProjectedExtent', 'TiltDis'])
    fig, axs = plt.subplots(2, 1, figsize=(11, 5), sharex=True,
        constrained_layout=True, gridspec_kw={'height_ratios': [2, 1]})
    axs[0].plot(daily.AgeDays, daily.MaxProjectedExtent, color='crimson',
                lw=1.8, label='Maximum projected extent')
    axs[0].plot(daily.AgeDays, daily.TiltDis, color='#222222',
                lw=1.8, label='Delta TiltDis')
    axs[0].set_title(f'Eddy {int(eddy)} · {len(valid)} paired days')
    axs[0].set_ylabel('Distance (km)')
    axs[0].legend()
    axs[1].plot(daily.AgeDays, daily.Difference, color='#3569a8', lw=1.5)
    axs[1].axhline(0, color='grey', lw=0.8)
    axs[1].set_ylabel('Extent − TiltDis (km)')
    axs[1].set_xlabel('Days since first available vertical profile')
    for ax in axs:
        ax.grid(alpha=0.2)
    if SAVE:
        fig.savefig(OUT / f'extent_vs_tilt_eddy_{int(eddy)}.png', dpi=180)
    plt.show()
    plt.close(fig)

summary = comparison.groupby('Eddy', sort=False).agg(
    PairedDays=('TiltDis', 'count'),
    MedianDifference_km=('Difference', 'median'),
    MedianAbsoluteDifference_km=('AbsoluteDifference', 'median'),
    MaxAbsoluteDifference_km=('AbsoluteDifference', 'max'))
display(summary.round(2))
if SAVE:
    comparison.to_csv(OUT / 'extent_vs_tilt_lifetimes.csv', index=False)
    summary.to_csv(OUT / 'extent_vs_tilt_summary.csv')

These are descriptive comparisons of a small sample. Maximum projected extent is a geometric comparison, not ground truth for delta tilt. Differences include temporal smoothing, profile curvature and line fitting. The exact reference and fit maximum depths are retained in `comparison` as `ReferenceDepthMax` and `FitDepthMax` for checking individual days.